# CcMart — Machine Learning Pipeline
**ITCS 6190/8190 Cloud Computing for Data Analysis**

Three MLlib models on the ingested Parquet tables:

1. **Random Forest — Session Conversion Prediction** — "will this session convert?" (binary classification on `session_funnel`).
2. **KMeans — Customer RFM Segmentation** — k=5 clusters over Recency / Frequency / Monetary.
3. **ALS — Product Recommender** — implicit collaborative filtering over customer × product interactions.

## Setup

In [1]:
import os

JAVA11 = "/usr/local/sdkman/candidates/java/11.0.30-ms"
os.environ["JAVA_HOME"] = JAVA11
clean_path = [p for p in os.environ["PATH"].split(":") if "sdkman" not in p and "jvm" not in p]
os.environ["PATH"] = f"{JAVA11}/bin:" + ":".join(clean_path)

# IMPORTANT: driver memory must be set BEFORE the JVM starts. In PySpark local
# mode, `.config("spark.driver.memory", ...)` in the builder is ignored — the
# JVM has already launched with the 1GB default. Use PYSPARK_SUBMIT_ARGS.
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 3g "
    "--conf spark.driver.maxResultSize=512m "
    "--packages org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StringIndexer, StandardScaler, OneHotEncoder
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator, BinaryClassificationEvaluator,
    ClusteringEvaluator, RegressionEvaluator
)

spark = SparkSession.builder \
    .appName('CcMart-ML') \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", os.environ.get("AWS_ACCESS_KEY_ID", "")) \
    .config("spark.hadoop.fs.s3a.secret.key", os.environ.get("AWS_SECRET_ACCESS_KEY", "")) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Driver memory:', spark.sparkContext._conf.get("spark.driver.memory", "default"))

BUCKET = os.environ.get("S3_BUCKET_PATH", "")
PROC = f"{BUCKET}/processed" if BUCKET else "../data/processed"

customers    = spark.read.parquet(f'{PROC}/customers_clean')
products     = spark.read.parquet(f'{PROC}/products_clean')
transactions = spark.read.parquet(f'{PROC}/transactions_exploded')
clicks       = spark.read.parquet(f'{PROC}/clickstream_clean')
sessions     = spark.read.parquet(f'{PROC}/session_funnel')
cart_events  = spark.read.parquet(f'{PROC}/cart_events')

print('MLlib ready. Loaded from', PROC)

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED
26/04/20 05:12:30 WARN Utils: Your hostname, codespaces-94354a resolves to a loopback address: 127.0.0.1; using 10.0.12.148 instead (on interface eth0)
26/04/20 05:12:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/codespace/.ivy2/cache
The jars for the packages stored in: /home/codespace/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-07fe51e8-fc02-4834-b90e-7f5640857e63;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 613ms :: artifacts dl 9ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlde

Driver memory: 3g


26/04/20 05:12:41 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


MLlib ready. Loaded from s3a://click-stream-s3//processed


## Model 1 — Random Forest: Session Conversion Prediction

**Target:** `converted` (1 if the session ended with a successful BOOKING, else 0) from `session_funnel`.

**Features:** `total_events`, `session_duration_mins`, `visited_homepage`, `did_search`, `added_to_cart`, `used_promo`, `num_keywords`, and one-hot-encoded `traffic_source`.

**Why Random Forest?** Handles mixed numeric + indicator features, captures non-linear interactions, robust to outliers, gives feature importance — a sensible baseline for conversion prediction.

In [2]:
# 895K sessions is more than we need for a demo — sample 150K for tractable RF
# training on the 8GB codespace.
SAMPLE_N = 150_000

data = (sessions
        .withColumn('num_keywords', F.size('keywords_searched'))
        .withColumn('label', F.col('converted').cast('double'))
        .fillna(0, subset=['session_duration_mins', 'num_keywords'])
        .na.drop(subset=['traffic_source'])
        .sample(fraction=SAMPLE_N / sessions.count(), seed=42))

print(f'Training rows (sampled): {data.count():,}')

traffic_idx = StringIndexer(inputCol='traffic_source', outputCol='traffic_idx',
                            handleInvalid='keep')
traffic_ohe = OneHotEncoder(inputCol='traffic_idx', outputCol='traffic_vec')
assembler = VectorAssembler(
    inputCols=['total_events', 'session_duration_mins',
               'visited_homepage', 'did_search', 'added_to_cart',
               'used_promo', 'num_keywords', 'traffic_vec'],
    outputCol='features', handleInvalid='skip'
)
rf = RandomForestClassifier(
    featuresCol='features', labelCol='label',
    numTrees=20, maxDepth=5, subsamplingRate=0.5, seed=42,
)

pipeline = Pipeline(stages=[traffic_idx, traffic_ohe, assembler, rf])
train, test = data.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)
preds = model.transform(test)

acc = MulticlassClassificationEvaluator(labelCol='label', metricName='accuracy').evaluate(preds)
f1  = MulticlassClassificationEvaluator(labelCol='label', metricName='f1').evaluate(preds)
auc = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC').evaluate(preds)
print(f'Accuracy : {acc:.4f}')
print(f'F1       : {f1:.4f}')
print(f'AUC-ROC  : {auc:.4f}')

rf_model = model.stages[-1]
feature_names = ['total_events', 'session_duration_mins', 'visited_homepage',
                 'did_search', 'added_to_cart', 'used_promo', 'num_keywords', 'traffic_vec']
importances = list(zip(feature_names, rf_model.featureImportances.toArray()))
print('\nFeature importances:')
for name, imp in sorted(importances, key=lambda x: -x[1]):
    print(f'  {name:<25} {imp:.4f}')

Training rows (sampled): 150,063


Accuracy : 0.9423
F1       : 0.9288
AUC-ROC  : 0.7659

Feature importances:
  added_to_cart             0.6355
  session_duration_mins     0.2204
  total_events              0.1130
  used_promo                0.0296
  num_keywords              0.0008
  did_search                0.0005
  traffic_vec               0.0000
  visited_homepage          0.0000


## Model 2 — KMeans Customer RFM Segmentation (k=5)

**Features (per customer, success-payment lines only):**
- `recency_days` = days since last successful booking (vs global max).
- `frequency`    = number of distinct bookings.
- `monetary`     = sum of `quantity * item_price`.

Features are `StandardScaler`-normalised before KMeans. k=5 is the canonical RFM segmentation size (Champions / Loyal / Potential / At Risk / Hibernating).

In [3]:
success_txns = transactions.filter(F.col('payment_status') == 'Success')

max_date = success_txns.agg(F.max('created_at')).first()[0]

rfm = success_txns.groupBy('customer_id').agg(
    F.datediff(F.lit(max_date).cast('timestamp'), F.max('created_at')).alias('recency'),
    F.countDistinct('booking_id').alias('frequency'),
    F.sum(F.col('quantity') * F.col('item_price')).alias('monetary'),
).na.drop()

assembler2 = VectorAssembler(inputCols=['recency', 'frequency', 'monetary'],
                             outputCol='raw_features')
scaler = StandardScaler(inputCol='raw_features', outputCol='features',
                        withStd=True, withMean=True)
kmeans = KMeans(featuresCol='features', predictionCol='cluster', k=5, seed=42, maxIter=20)

pipeline2 = Pipeline(stages=[assembler2, scaler, kmeans])
model2 = pipeline2.fit(rfm)
preds2 = model2.transform(rfm)

silhouette = ClusteringEvaluator(
    featuresCol='features', predictionCol='cluster', metricName='silhouette'
).evaluate(preds2)
print(f'Silhouette score: {silhouette:.4f}\n')

# Cluster profile — what each segment looks like on raw RFM
preds2.groupBy('cluster').agg(
    F.count('*').alias('n_customers'),
    F.round(F.avg('recency'), 1).alias('avg_recency_days'),
    F.round(F.avg('frequency'), 2).alias('avg_frequency'),
    F.round(F.avg('monetary'), 0).alias('avg_monetary'),
).orderBy('cluster').show(truncate=False)

26/04/20 05:14:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Silhouette score: 0.7129



+-------+-----------+----------------+-------------+------------+
|cluster|n_customers|avg_recency_days|avg_frequency|avg_monetary|
+-------+-----------+----------------+-------------+------------+
|0      |32566      |210.3           |6.11         |3134881.0   |
|1      |8850       |36.1            |33.76        |1.8694554E7 |
|2      |394        |30.2            |210.31       |1.15918207E8|
|3      |5899       |1324.4          |1.18         |636456.0    |
|4      |2533       |23.6            |90.23        |4.9822314E7 |
+-------+-----------+----------------+-------------+------------+



## Model 3 — ALS Collaborative Filtering Recommender

**Interaction matrix (customer × product):**
- Successful purchase (transaction_exploded) → rating 5
- Added to cart but never bought (cart_events joined to transaction customer) → rating 3

We collapse duplicates with `MAX(rating)` so a purchased item is never under-weighted by an earlier cart event.

`ALS(implicitPrefs=True)` treats the rating as a confidence weight and learns latent user/item factors.

In [ ]:
# Explicit interactions from successful purchases
purchase_interactions = transactions \
    .filter((F.col('payment_status') == 'Success') & F.col('product_id').isNotNull()) \
    .select('customer_id', 'product_id') \
    .withColumn('rating', F.lit(5.0))

# Implicit interactions from cart adds — get customer via session → transaction mapping
session_to_customer = transactions.select('session_id', 'customer_id').dropDuplicates()
cart_interactions = cart_events \
    .join(session_to_customer, on='session_id', how='inner') \
    .filter(F.col('product_id').isNotNull()) \
    .select('customer_id', 'product_id') \
    .withColumn('rating', F.lit(3.0))

interactions = purchase_interactions.unionByName(cart_interactions) \
    .groupBy('customer_id', 'product_id') \
    .agg(F.max('rating').alias('rating')) \
    .withColumn('customer_id', F.col('customer_id').cast('int')) \
    .withColumn('product_id',  F.col('product_id').cast('int')) \
    .repartition(8)

n_int = interactions.count()
n_users = interactions.select('customer_id').distinct().count()
n_items = interactions.select('product_id').distinct().count()
print(f'Interactions : {n_int:>10,}')
print(f'Users        : {n_users:>10,}')
print(f'Items        : {n_items:>10,}')
print(f'Sparsity     : {(1 - n_int / (n_users * n_items)) * 100:.2f}%\n')

train3, test3 = interactions.randomSplit([0.8, 0.2], seed=42)

# Lighter ALS — rank 8, 5 iterations fits comfortably in 4GB driver.
als = ALS(
    userCol='customer_id', itemCol='product_id', ratingCol='rating',
    rank=8, maxIter=5, regParam=0.1,
    implicitPrefs=True, nonnegative=True,
    coldStartStrategy='drop', seed=42,
)
als_model = als.fit(train3)

rmse = RegressionEvaluator(
    labelCol='rating', predictionCol='prediction', metricName='rmse'
).evaluate(als_model.transform(test3))
print(f'ALS RMSE (test): {rmse:.4f}\n')

# Recommend for a sample of 200 users (full recommendForAllUsers is 100K users
# × top-5 = 500K-row shuffle that blows past driver memory).
sample_users = (interactions
                .select('customer_id').distinct()
                .limit(200))
recs = als_model.recommendForUserSubset(sample_users, 5)
recs_flat = recs.select(
    F.col('customer_id'),
    F.explode('recommendations').alias('rec')
).select(
    'customer_id',
    F.col('rec.product_id').alias('product_id'),
    F.round(F.col('rec.rating'), 3).alias('predicted_score'),
).join(
    products.select('product_id', 'productDisplayName', 'masterCategory'),
    on='product_id', how='left'
).orderBy('customer_id', F.desc('predicted_score'))

print('Sample recommendations (200 users × top-5):')
recs_flat.show(15, truncate=False)

Interactions :  1,889,569
Users        :     50,705
Items        :     44,446
Sparsity     : 99.92%



ALS RMSE (test): 4.3609

Sample recommendations (200 users × top-5):


+----------+-----------+---------------+----------------------------------------------------+--------------+
|product_id|customer_id|predicted_score|productDisplayName                                  |masterCategory|
+----------+-----------+---------------+----------------------------------------------------+--------------+
|14804     |67         |0.013          |United Colors of Benetton Women Solid Black Handbags|Accessories   |
|7804      |67         |0.011          |Puma Men's Benecio Mid Leather Black White Shoe     |Footwear      |
|20460     |67         |0.011          |Baggit Women Babe Opart White Handbag               |Accessories   |
|18568     |67         |0.01           |Puma Men Benecio Canvas White Casual Shoes          |Footwear      |
|41679     |67         |0.01           |Catwalk Women Grey Heels                            |Footwear      |
|14804     |161        |0.069          |United Colors of Benetton Women Solid Black Handbags|Accessories   |
|7804      |161    

: 

: 

## Summary

| # | Model | Task | Key Metric |
|---|-------|------|------------|
| 1 | Random Forest | Session conversion classification | Accuracy / F1 / AUC |
| 2 | KMeans (k=5) | Customer RFM segmentation | Silhouette |
| 3 | ALS (implicit) | Product recommender | RMSE + top-5 recs |

**Business framing (matches the pitch):**
- Model 1 → trigger real-time discount pop-ups for likely-to-bounce users.
- Model 2 → identify Champions / At Risk / Hibernating cohorts for lifecycle marketing.
- Model 3 → "customers like you also bought…" carousel, driving AOV uplift.

## Run the full ML script
```bash
python ../src/ml_pipeline.py
```
> Note: `src/ml_pipeline.py` still targets the legacy sample-data schema and will need a re-sync before it runs against the new Parquet tables. The notebook above is the authoritative working version.